# ML-04 — Search Intelligence Data Contract



## Setup — connect to the FlyRank warehouse on Hugging Face

DuckDB + `hf://` + `CREATE SECRET` for the token, reading Parquet directly off Hugging Face —
no full download. Requires an `HF_TOKEN` Colab secret (key icon, left sidebar); accept the gate
on the dataset page first.

In [1]:
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except ImportError:
    import os
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "Add HF_TOKEN as a Colab secret (key icon, left sidebar) before running -- accept the dataset gate first."

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
print("Connected. Token loaded from secret, never printed.")


Connected. Token loaded from secret, never printed.


**Point at the table, let DuckDB find the month partitions itself.** `fact_content_daily_performance`
is Hive-partitioned by `month=YYYY-MM` folders; `dim_clients` is a single flat file at the repo
root (`dim_clients.parquet`, confirmed via `list_repo_files`, not a folder). `2026-03` is the
mid-panel month; `2026-06` (the `_sample` table) stays sealed, per the warning on the card.


In [2]:
rel = "hf://datasets/FlyRank/internship-warehouse"
REPORT_MONTH = "2026-03"  # mid-panel month -- NOT the sealed final month (2026-06 / the _sample table)

con.sql(f"""
CREATE OR REPLACE VIEW fact_all AS
SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)
""")

con.sql("SELECT DISTINCT month FROM fact_all ORDER BY month").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┐
│  month  │
│ varchar │
├─────────┤
│ 2025-01 │
│ 2025-02 │
│ 2025-03 │
│ 2025-04 │
│ 2025-05 │
│ 2025-06 │
│ 2025-07 │
│ 2025-08 │
│ 2025-09 │
│ 2025-10 │
│ 2025-11 │
│ 2025-12 │
│ 2026-01 │
│ 2026-02 │
│ 2026-03 │
│ 2026-04 │
│ 2026-05 │
│ 2026-06 │
├─────────┤
│ 18 rows │
└─────────┘



In [3]:
con.sql(f"""
CREATE OR REPLACE VIEW fact_march AS
SELECT * FROM fact_all WHERE month = '{REPORT_MONTH}'
""")

con.sql(f"CREATE OR REPLACE VIEW dim_clients AS SELECT * FROM read_parquet('{rel}/dim_clients.parquet')")

con.sql("SELECT COUNT(*) AS n_rows FROM fact_march").show()


┌─────────┐
│ n_rows  │
│  int64  │
├─────────┤
│ 9841378 │
└─────────┘



**Real columns, confirmed via `DESCRIBE`** (not guessed): the identifier columns are
`client_hash_id` / `content_hash_id`, not `client_id` / `content_id`. There is no raw `ctr`,
`engagement_rate`, or `scroll_rate` column — those get computed below from raw counts
(`gsc_clicks`/`gsc_impressions`, `ga4_engaged_sessions`/`ga4_sessions`, `scroll_events`/`ga4_pageviews`).
Both `gsc_data_available` and `ga4_data_available` exist as real boolean flags.


In [4]:
con.sql("DESCRIBE fact_march").show(max_rows=60)


┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date              │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gs

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item, on one calendar day, for one client** —
`(client_hash_id, content_hash_id, report_date)` — from `fact_content_daily_performance`,
restricted to `report_date` in March 2026 (the mid-panel month, per the card's warning to avoid
the sealed final month). This is a content-day grain: the same content item contributes one row
per day it has search data that month, not one row total.


In [5]:
con.sql("""
SELECT MIN(report_date) AS first_date, MAX(report_date) AS last_date, COUNT(*) AS n_rows
FROM fact_march
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬─────────┐
│ first_date │ last_date  │ n_rows  │
│    date    │    date    │  int64  │
├────────────┼────────────┼─────────┤
│ 2026-03-01 │ 2026-03-31 │ 9841378 │
└────────────┴────────────┴─────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature** — `gsc_avg_position`, `gsc_impressions`, `gsc_clicks`, and three computed ratios:
  `ctr_mar` (`gsc_clicks`/`gsc_impressions`), `engagement_rate_mar`
  (`ga4_engaged_sessions`/`ga4_sessions`, GA4-available rows only), `active_days_mar`. All
  measured about the content item *during* March, so all knowable by the end of March — before
  any April outcome exists.
- **Label / proxy** — not a column in this table at all. "Did this content decline" only exists
  by comparing March's aggregate clicks to April's, which needs *next* month's data. Below, this
  gets built on purpose, once, only to demonstrate the leak trap — it is not a real feature.
- **Context** — `client_hash_id`, `content_hash_id`, `report_date`. Identifiers for
  grouping/joining/filtering only — never fed as raw values into a model.
- **Excluded** — the raw `gsc_data_available` / `ga4_data_available` flags as *features* (they
  signal which measurement system was live, not content quality — used only as WHERE filters);
  `client_has_gsc` / `client_has_ga4` (client-level access flags, not content signal); anything
  from April or later (future information relative to a March decision moment — this is exactly
  the trap below).


In [6]:
feature_cols  = ["gsc_avg_position", "gsc_impressions", "gsc_clicks", "ctr_mar (computed)", "engagement_rate_mar (computed)", "active_days_mar (computed)"]
context_cols  = ["client_hash_id", "content_hash_id", "report_date"]
excluded_cols = ["gsc_data_available (raw, as a feature)", "ga4_data_available (raw, as a feature)", "client_has_gsc / client_has_ga4", "any April+ column"]

print("feature :", feature_cols)
print("context :", context_cols)
print("excluded:", excluded_cols)
print("label/proxy: not present in this table -- engineered downstream, see the trap section")


feature : ['gsc_avg_position', 'gsc_impressions', 'gsc_clicks', 'ctr_mar (computed)', 'engagement_rate_mar (computed)', 'active_days_mar (computed)']
context : ['client_hash_id', 'content_hash_id', 'report_date']
excluded: ['gsc_data_available (raw, as a feature)', 'ga4_data_available (raw, as a feature)', 'client_has_gsc / client_has_ga4', 'any April+ column']
label/proxy: not present in this table -- engineered downstream, see the trap section


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three facts, in order: the grain, the slice's size and date span, and availability filtered
with `IS TRUE`. Then five features, then the deliberate leak.


### 3a. Fact 1 — the grain really is (client_hash_id, content_hash_id, report_date)

In [7]:
grain_violations = con.sql("""
SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
FROM fact_march
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("rows violating the stated grain (should be empty):")
grain_violations


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows violating the stated grain (should be empty):


,client_hash_id,content_hash_id,report_date,c


### 3b. Fact 2 — row count and date span of this slice

In [8]:
con.sql("""
SELECT
  COUNT(*)                         AS n_rows,
  COUNT(DISTINCT content_hash_id)  AS n_content_items,
  COUNT(DISTINCT client_hash_id)   AS n_clients,
  MIN(report_date)                 AS first_date,
  MAX(report_date)                 AS last_date
FROM fact_march
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬─────────────────┬───────────┬────────────┬────────────┐
│ n_rows  │ n_content_items │ n_clients │ first_date │ last_date  │
│  int64  │      int64      │   int64   │    date    │    date    │
├─────────┼─────────────────┼───────────┼────────────┼────────────┤
│ 9841378 │          331437 │        55 │ 2026-03-01 │ 2026-03-31 │
└─────────┴─────────────────┴───────────┴────────────┴────────────┘



### 3c. Fact 3 — availability, filtered with IS TRUE

In [9]:
con.sql("""
SELECT
  COUNT(*)                                                      AS total_rows,
  SUM(CASE WHEN gsc_data_available IS TRUE  THEN 1 ELSE 0 END)  AS gsc_true,
  SUM(CASE WHEN gsc_data_available IS FALSE THEN 1 ELSE 0 END)  AS gsc_false,
  SUM(CASE WHEN gsc_data_available IS NULL  THEN 1 ELSE 0 END)  AS gsc_null,
  SUM(CASE WHEN ga4_data_available IS TRUE  THEN 1 ELSE 0 END)  AS ga4_true,
  SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END)  AS ga4_false,
  SUM(CASE WHEN ga4_data_available IS NULL  THEN 1 ELSE 0 END)  AS ga4_null
FROM fact_march
""").show()

print("rows that survive the safe filter (IS TRUE, not = TRUE):")
con.sql("SELECT COUNT(*) AS n_survives FROM fact_march WHERE ga4_data_available IS TRUE").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────┬───────────┬──────────┬──────────┬───────────┬──────────┐
│ total_rows │ gsc_true │ gsc_false │ gsc_null │ ga4_true │ ga4_false │ ga4_null │
│   int64    │  int128  │  int128   │  int128  │  int128  │  int128   │  int128  │
├────────────┼──────────┼───────────┼──────────┼──────────┼───────────┼──────────┤
│    9841378 │  3611061 │   6230317 │        0 │   413966 │   6408671 │  3018741 │
└────────────┴──────────┴───────────┴──────────┴──────────┴───────────┴──────────┘

rows that survive the safe filter (IS TRUE, not = TRUE):
┌────────────┐
│ n_survives │
│   int64    │
├────────────┤
│     413966 │
└────────────┘



### 3d. Five features — each knowable at the decision moment (end of March)

1. **`avg_position_mar`** — knowable because GSC position is measured continuously through
   March; no future data needed.
2. **`impressions_mar`** — knowable because it's a straight sum of March impressions.
3. **`ctr_mar`** — knowable because it's computed from March `gsc_clicks` / March
   `gsc_impressions` only.
4. **`engagement_rate_mar`** — knowable only where `ga4_data_available IS TRUE`; computed as
   `ga4_engaged_sessions` / `ga4_sessions` summed over March GA4-available rows, no later data
   touched.
5. **`active_days_mar`** — knowable because it just counts March days with any impressions —
   a same-month freshness/activity signal.


In [10]:
features_df = con.sql("""
SELECT
  client_hash_id,
  content_hash_id,
  AVG(gsc_avg_position)                                                              AS avg_position_mar,
  SUM(gsc_impressions)                                                               AS impressions_mar,
  SUM(gsc_clicks)                                                                    AS clicks_mar,
  CASE WHEN SUM(gsc_impressions) > 0
       THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) ELSE NULL END               AS ctr_mar,
  CASE WHEN SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) > 0
       THEN SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) * 1.0
            / SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END)
       ELSE NULL END                                                                AS engagement_rate_mar,
  COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)                    AS active_days_mar
FROM fact_march
GROUP BY client_hash_id, content_hash_id
""").df()

print(features_df.shape)
features_df.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 8)


,client_hash_id,content_hash_id,avg_position_mar,impressions_mar,clicks_mar,ctr_mar,engagement_rate_mar,active_days_mar
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.209549,6523.0,7.0,0.001073,0.0,31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2.987198,453.0,0.0,0.000000,NaN,31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,6.724039,5630.0,6.0,0.001066,0.0,31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,7.244844,4944.0,13.0,0.002629,0.0,31
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,14.432540,42.0,0.0,0.000000,0.0,21


### 3e. The trap — add one label-derived column on purpose, watch the score jump, then remove it

To even have a "declining" label we need *next* month's outcome, which by definition can't be
known in March. Pulling April clicks just once here, only to build the trap and the proxy label
— not as a real feature.


In [11]:
APRIL_MONTH = "2026-04"

con.sql(f"""
CREATE OR REPLACE VIEW fact_april AS
SELECT * FROM fact_all WHERE month = '{APRIL_MONTH}'
""")

label_df = con.sql("""
SELECT content_hash_id, SUM(gsc_clicks) AS clicks_apr
FROM fact_april
GROUP BY content_hash_id
""").df()

merged = features_df.merge(label_df, on="content_hash_id", how="inner")
merged["is_declining"] = (merged["clicks_apr"] < merged["clicks_mar"]).astype(int)

print("rows with both March features and an April outcome:", len(merged))
print("share declining:", round(merged["is_declining"].mean(), 3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows with both March features and an April outcome: 331436
share declining: 0.136


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["avg_position_mar", "impressions_mar", "clicks_mar", "ctr_mar", "engagement_rate_mar", "active_days_mar"]

X = merged[honest_features].fillna(0)
y = merged["is_declining"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
honest_auc = roc_auc_score(yte, LogisticRegression(max_iter=1000).fit(Xtr, ytr).predict_proba(Xte)[:, 1])

print("HONEST AUC (March-only features):", round(honest_auc, 3))


HONEST AUC (March-only features): 0.938


In [13]:
# THE TRAP -- smuggle April clicks in as if it were just another feature
merged["clicks_apr_LEAKED"] = merged["clicks_apr"]

leaked_features = honest_features + ["clicks_apr_LEAKED"]
Xl = merged[leaked_features].fillna(0)

Xtr, Xte, ytr, yte = train_test_split(Xl, y, test_size=0.3, random_state=42, stratify=y)
leaked_auc = roc_auc_score(yte, LogisticRegression(max_iter=1000).fit(Xtr, ytr).predict_proba(Xte)[:, 1])

print("LEAKED AUC (April clicks smuggled in as a 'feature'):", round(leaked_auc, 3))
print("HONEST AUC (no leak):", round(honest_auc, 3))
print("jump from the leak:", round(leaked_auc - honest_auc, 3))
print()
print("This 'feature' is the other half of the label -- of course it looks almost perfect. Removing it now.")

del merged["clicks_apr_LEAKED"]
print()
print("Kept, and reported going forward: the HONEST AUC =", round(honest_auc, 3))


LEAKED AUC (April clicks smuggled in as a 'feature'): 1.0
HONEST AUC (no leak): 0.938
jump from the leak: 0.062

This 'feature' is the other half of the label -- of course it looks almost perfect. Removing it now.

Kept, and reported going forward: the HONEST AUC = 0.938


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: history depth is uneven across clients, so this slice under-represents newer
clients.** `dim_clients.gsc_data_start` varies per client — some clients have search history
going back to the panel's start, others started syncing mid-panel, so their March row counts are
thin or zero through no fault of their content. A model trained on this slice will implicitly
learn more about long-tenured clients than new ones. Separately, GA4 rows are zero-filled (not
truly zero-engagement) before each client's `ga4_data_start`, which is exactly why
`engagement_rate_mar` above is only averaged where `ga4_data_available IS TRUE` rather than
naively over every row.


In [14]:
con.sql("""
SELECT c.gsc_data_start, COUNT(*) AS n_march_rows
FROM fact_march f
JOIN dim_clients c USING (client_hash_id)
GROUP BY c.gsc_data_start
ORDER BY c.gsc_data_start
LIMIT 15
""").show()


┌────────────────┬──────────────┐
│ gsc_data_start │ n_march_rows │
│      date      │    int64     │
├────────────────┼──────────────┤
│ 2025-01-27     │       235538 │
│ 2025-02-11     │       869640 │
│ 2025-03-11     │       335379 │
│ 2025-06-07     │       756660 │
│ 2025-06-18     │       126512 │
│ 2025-06-21     │       484091 │
│ 2025-06-29     │       316610 │
│ 2025-07-01     │       988497 │
│ 2025-07-06     │        36538 │
│ 2025-07-07     │        53165 │
│ 2025-07-17     │        36828 │
│ 2025-07-21     │        12369 │
│ 2025-07-28     │        11067 │
│ 2025-07-29     │        17825 │
│ 2025-09-24     │      1566856 │
├────────────────┴──────────────┤
│ 15 rows             2 columns │
└───────────────────────────────┘

